# Investigating duplicate names across languages
Featuring
- name normalization
- exact comparison
- tokenization-based comparison
- similarity-based comparison

In [2]:
# data manipulation libraries
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from itertools import combinations
from typing import Iterable, Any, Union, List

# text string operations libraries
import unicodedata
from faker import Faker
import string
import re
from rapidfuzz import fuzz

# streamlined notebook display (not otherwise recommended)
import warnings
warnings.filterwarnings("ignore")

In [3]:
def deluxe_value_counts(df: pd.DataFrame, column: Union[str, int]) -> pd.DataFrame:
    """
    Generate enhanced value counts with both absolute frequencies and percentages.

    This function creates a "deluxe" version of pandas' value_counts() that displays
    both the count of occurrences (#) and percentage (%) for each unique value in
    the specified column. Includes NaN values in the count and rounds percentages to
    1 decimal place.

    Parameters:
    -----------
    df : pd.DataFrame
        Input DataFrame containing the column to analyze.
    column : Union[str, int]
        Name of the column (string) or column index (integer) to compute value counts for.

    Returns:
    --------
    pd.DataFrame
        MultiIndex column DataFrame with two columns:
        - '#': Absolute counts for each unique value (including NaN)
        - '%': Percentages of total occurrences, rounded to 1 decimal place

    Raises:
    -------
    KeyError
        If the specified column name or index does not exist in the DataFrame.
    ValueError
        If the column is empty or contains no valid data.

    Examples:
    ---------
    >>> data = pd.DataFrame({'category': ['A', 'B', 'A', 'A', None, 'B']})
    >>> deluxe_value_counts(data, 'category')
               #    %
    A          3  50.0
    B          2  33.3
    NaN        1  16.7

    Notes:
    ------
    - NaN values are always included (dropna=False)
    - Percentages are calculated relative to total observations (including NaN)
    - Percentages are rounded to 1 decimal place for readability
    - Original function preserves pandas' default sorting by count (descending)
    """
    counts = df[column].value_counts(dropna=False)
    percentages = df[column].value_counts(normalize=True, dropna=False).mul(100).round(1)
    deluxe_count = pd.concat([counts, percentages], axis=1, keys=['#', '%'])
    return deluxe_count

def tokenize(name: str) -> List[str]:
    """
    Tokenize a name-like string into word tokens

    Args:
        name: The input string to tokenize.

    Returns:
        Tokens extracted from the input string.

    Example:
        >>> tokenize("John Smith")
        ['John', 'Smith']
    """
    return re.findall(r"\w+", name)

Because this notebook will be leveraging the Faker library to generate the data that will be later used, normal data exploration and validation steps are skipped here.

In [5]:
# instantiating an instance of Faker with multiple locales
# some locales do not have diacritics while others do
fake = Faker(
    [
        'en_CA', 'fr_FR', 'es_MX', 'pt_BR', 'de_DE', 
        'zu_ZA', 'yo_NG', 'tl_PH', 'sk_SK', 'ro_RO'
    ]
)

# generating randomized names across locales
def create_employees(num_employees: int):
    employee_list = []
    for i in range(num_employees):
        employee = {}
        employee['ee#'] = 10000000+i
        employee['raw_name'] = fake.name()
        employee_list.append(employee)
    return pd.DataFrame(employee_list)

name_records = create_employees(5000)
name_records.sample(10)

,ee#,raw_name
77,10000077,Ogbeni Ibidapo Soyinka
1339,10001339,Uli Caspar
2947,10002947,Severín Král
2004,10002004,Oba Tade Ogindan
933,10000933,Thandiwe Mpila-Yei
3046,10003046,pani Frederika Hudecová
2826,10002826,Sibongile Mbuyise
3269,10003269,Zinhle Mphephethwa
1041,10001041,Luna Barros
1066,10001066,Marianne Imbert


In [6]:
# normalizing names to enable comparison at scale
translator = str.maketrans('', '', string.punctuation) # creating a table of punctuation
name_records['norm_names'] = name_records['raw_name'].apply(lambda x: 
                                          unicodedata.normalize('NFKD', x) # decompose diacritics
                                          .encode('ascii', errors= 'ignore') # recompose without diacritics
                                          .decode('utf-8') # back to string format
                                          .casefold() # remove case, works on letters ignored in previous steps
                                          .strip().replace('\s+', ' ') # standardize whitespace
                                          .translate(translator) # remove punctuation using the table above
                                         )

In [7]:
# identifying exact duplicates
name_counts = Counter(name_records['norm_names']) # count occurrences of each name
exact_duplicates = (
    pd.DataFrame.from_dict(name_counts, orient= "index")
    .reset_index()
    .rename(columns= {"index": "name", 0: "exact_count"})
    .query("exact_count > 1") # create a df where a name appears more than once
)

print(exact_duplicates.shape)
exact_duplicates

(14, 2)


,name,exact_count
350,daniela machado,2
984,ronald garcia,2
1152,andile nyongwana,2
1243,abayomi sobowale,2
1447,vitor moura,2
1483,michael obrien,2
1505,viorela nemes,2
1663,dra heloisa pinto,2
2741,kevin da luz,2
3063,melanie rodriguez,2


In [8]:
# identifying potential duplicates due to inversion of name parts
# e.g., middle name in place of first name and vice versa
token_sets = {
    name: frozenset(tokenize(name)) # frozen sets is used for uniqueness and immutability
    for name in name_records['norm_names'] # extracting the tokens from each name
}

'''
A best practice here would be to include a validation step to handle 
    missing or NaN values, which is not present here because the data
    generation process causes an absence of the missing values that 
    would be a feature of real world data.
'''

groups = defaultdict(list)
for name, tokens in token_sets.items():
    groups[tokens].append(name) # groups names based on tokens they contain

token_based_matches = (
    pd.DataFrame( # displaying the groupings as a df
        [
            {
            "token_group": tokens, # groups of tokens (e.g., ['David', 'James'])
            "names": names, # names associated with the token group (e.g., 'David James', 'James David')
            "name_count": len(names) # count of unique names associated with the same token group
            }
            for tokens, names in groups.items()
            if len(names) > 1 # include where there is more than one name associated with a token group
        ]
    )
)

if not token_based_matches.empty: # if there are matches, then proceed
    # get the pairs of matches in the dataset for each token group
    pairs = []
    
    for row in token_based_matches.itertuples():
        for a, b in combinations(row.names, 2):
            pairs.append({
                "token_group": row.token_group,
                "name_1": a,
                "name_2": b
            })
    
    token_name_pairs = pd.DataFrame(pairs)
    token_name_pairs['token_group'] = ( # overwriting the column for readability
        token_name_pairs['token_group'] # skip this step or create a new column
            .apply(lambda s: ", ".join(sorted(s))) # to maintain original token groups
    )

    '''
    While redundant for the remaining notebook, pair-wise analysis
    presents a valuable perspective with real-world data to understand patterns
    in the pairs of names extracted from the dataset.
    '''
    
    # get a paired name for each name in the dataset for each token group
    merge_ready_token_matches = (
        token_based_matches
            .explode("names")
            .rename(columns= {"names": "name"})
            .reset_index(drop= True)
    )
    merge_ready_token_matches['token_group'] = ( # same overwrite as above for readability
        merge_ready_token_matches['token_group']
            .apply(lambda s: ", ".join(sorted(s)))
    )

else: 
    d = {
        "token_group": np.nan,
        "name": name_records['norm_names'],
        "name_count": np.nan
    }
    merge_ready_token_matches = pd.DataFrame(data= d)
    print(f"No token based potential duplicates have been found.")

No token based potential duplicates have been found.


In [9]:
# identifying potential duplicates by similarity (misspellings, typos, etc.)
def blockingKey(name):
    parts = name.split()
    if len(parts) == 0:
        return None
    return (parts[0][0] if parts[0] else "", parts[-1][0] if parts [-1] else "")

blocks = defaultdict(list)
for name in name_records['norm_names']:
    blocks[blockingKey(name)].append(name) 
    # creating blocks of names that share initials for better processing management
    # because of the combinatorial explosion problem

potential_matches = []

for block in blocks.values(): # sort the names alphabetically within each block 
    names = sorted(set(block))
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            score = fuzz.token_sort_ratio(a, b) # then compare the names with the normalized Indel similarity
            if score > 90: # keep only the strong matches
                potential_matches.append(
                        {
                        "name_a": a,
                        "name_b": b,
                        "similarity": score
                        }
                )

potential_matches = pd.DataFrame(potential_matches).sort_values(by= 'similarity', 
                                                                ascending= False)
print(potential_matches.shape)
potential_matches.head()

(15, 3)


,name_a,name_b,similarity
13,lara da cunha,laura da cunha,96.296296
6,luana barros,luna barros,95.652174
2,ekundayo olabode,ekundayo olubode,93.750000
4,adina gheorghiu,arina gheorghiu,93.333333
7,simina dochioiu,simona dochioiu,93.333333


In [10]:
# preparing a dataframe ready for merging back with original list
df_forward = potential_matches.rename(
    columns= {
        'name_a': 'name',
        'name_b': 'paired_name'
    }
)
df_backward = potential_matches.rename(
    columns= {
        'name_a': 'paired_name',
        'name_b': 'name'
    }
)

merge_ready_potential_matches = (
    pd.concat([df_forward, df_backward], ignore_index= True)
)

print(merge_ready_potential_matches.shape)
merge_ready_potential_matches.head()

(30, 3)


,name,paired_name,similarity
0,lara da cunha,laura da cunha,96.296296
1,luana barros,luna barros,95.652174
2,ekundayo olabode,ekundayo olubode,93.750000
3,adina gheorghiu,arina gheorghiu,93.333333
4,simina dochioiu,simona dochioiu,93.333333


In [11]:
# merging all dataframes together
name_records = name_records.merge(
    exact_duplicates,
    how= 'left',
    left_on= "norm_names",
    right_on= "name"
)

name_records = name_records.merge(
    merge_ready_token_matches,
    how= 'left',
    left_on= 'norm_names',
    right_on= 'name'
)

name_records = name_records.merge(
    merge_ready_potential_matches,
    how= 'left',
    left_on= 'norm_names',
    right_on= 'name'
)

# cleaning and visualization steps
name_records.drop(labels= ['name', 'name_x', 'name_y', 'name_count'], 
                  axis= 1, inplace=True)
name_records.sample(10)

,ee#,raw_name,norm_names,exact_count,token_group,paired_name,similarity
1661,10001653,Michael Rodriguez DDS,michael rodriguez dds,NaN,NaN,NaN,NaN
1750,10001741,Anežka Jedľovská,anezka jedlovska,NaN,NaN,NaN,NaN
4414,10004389,Casandra Mazilescu,casandra mazilescu,NaN,NaN,NaN,NaN
926,10000924,Tammy Stanley,tammy stanley,NaN,NaN,NaN,NaN
4936,10004908,Rainer Baum,rainer baum,NaN,NaN,NaN,NaN
2740,10002729,Camila Nogueira,camila nogueira,NaN,NaN,NaN,NaN
1759,10001750,Gugu Bhensela,gugu bhensela,NaN,NaN,NaN,NaN
418,10000417,Gbowoade Odunsi,gbowoade odunsi,NaN,NaN,NaN,NaN
4117,10004093,Dipl.-Ing. Rosita Klotz,dipling rosita klotz,NaN,NaN,NaN,NaN
4244,10004219,Fikayomi Adesiyan,fikayomi adesiyan,NaN,NaN,NaN,NaN


In [12]:
# identifying the name is a duplicate of any sort
name_records['concat_layers'] = (
    name_records[['exact_count', 'token_group', 'paired_name', 'similarity']]
    .apply(lambda x: '|'.join(x.dropna().astype(str)), axis= 1)
)

name_records['isLikelyDuplicate'] = (
    name_records['concat_layers'].apply(lambda x:
                                        False if x == "" else True
                                       )
)

name_records.drop(labels= ['concat_layers'], 
                       axis= 1, 
                       inplace= True)

# identifying the type of duplicate
name_records['duplicate_type'] = (
    name_records.apply(
        lambda x:
        'exact' if pd.notna(x['exact_count'])
        else 'token' if pd.notna(x['token_group'])
        else 'similar' if pd.notna(x['paired_name'])
        else 'not_duplicate' if x['isLikelyDuplicate'] == False
        else 'check',
        axis= 1
    )
)

name_records.sample(10)

,ee#,raw_name,norm_names,exact_count,token_group,paired_name,similarity,isLikelyDuplicate,duplicate_type
3518,10003502,Amy Watson,amy watson,NaN,NaN,NaN,NaN,False,not_duplicate
3528,10003512,Kabiyesi Sijuade Bakare,kabiyesi sijuade bakare,NaN,NaN,NaN,NaN,False,not_duplicate
3876,10003856,Claude Bourgeois,claude bourgeois,NaN,NaN,NaN,NaN,False,not_duplicate
3872,10003852,Christian Johnson,christian johnson,NaN,NaN,NaN,NaN,False,not_duplicate
4686,10004659,Mtro. Pedro Olivo,mtro pedro olivo,NaN,NaN,NaN,NaN,False,not_duplicate
417,10000416,S'fiso Jokiwe-Mangena,sfiso jokiwemangena,NaN,NaN,NaN,NaN,False,not_duplicate
1854,10001845,Marcela Hugo Partida,marcela hugo partida,NaN,NaN,NaN,NaN,False,not_duplicate
4762,10004734,Prof. Aleksej Oestrovsky MBA.,prof aleksej oestrovsky mba,NaN,NaN,NaN,NaN,False,not_duplicate
3588,10003571,Maximilián Kočko,maximilian kocko,NaN,NaN,NaN,NaN,False,not_duplicate
3247,10003233,Folashade Olaiya,folashade olaiya,NaN,NaN,NaN,NaN,False,not_duplicate


In [13]:
# how many duplicates of each type are in the dataframe
deluxe_value_counts(df= name_records, column= 'duplicate_type')

,#,%
duplicate_type,,
not_duplicate,4943,98.3
exact,56,1.1
similar,30,0.6
